In [ ]:
!pip -q install -U datasets transformers==4.46.3 accelerate==1.1.1 sentencepiece evaluate sacrebleu jiwer huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.6 MB/s eta 0:00:00


In [ ]:
import os, json, zipfile, glob, re, random
import numpy as np
import torch

from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)

LANGS = ["asm", "hin", "mar"]
LANG_TOKEN = {"asm": "<2asm>", "hin": "<2hin>", "mar": "<2mar>"}

MODEL_NAME = "google/mt5-small"

MAX_SRC = 32
MAX_TGT = 32

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

torch: 2.10.0+cu128 cuda: True


In [ ]:
def _extract_zip_once(lang: str, root="/content/aksharantar_zips"):
    """
    Downloading ai4bharat/Aksharantar <lang>.zip and extracts it locally once.
    """
    os.makedirs(root, exist_ok=True)
    zip_path = hf_hub_download(
        repo_id="ai4bharat/Aksharantar",
        filename=f"{lang}.zip",
        repo_type="dataset",
    )
    out_dir = os.path.join(root, lang)
    os.makedirs(out_dir, exist_ok=True)
    marker = os.path.join(out_dir, ".extracted")

    if not os.path.exists(marker):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(out_dir)
        open(marker, "w").close()

    return out_dir

def _find_split_file(extract_dir: str, split: str):
    """
    Finds a JSONL file for split under extracted dataset.
    Handles valid/val/validation.
    """
    patterns = [f"*{split}*.json"]
    if split in ["valid", "validation"]:
        patterns += ["*val*.json", "*valid*.json", "*validation*.json"]

    for pat in patterns:
        cands = glob.glob(os.path.join(extract_dir, "**", pat), recursive=True)
        if cands:
            cands = sorted(cands, key=len)
            return cands[0]

    raise FileNotFoundError(f"Could not find split='{split}' under {extract_dir}")

def load_aksharantar_jsonl(lang: str, split: str):
    """
    Returns list of {"src": ..., "tgt": ...} from JSONL.
    Expected keys: 'english word', 'native word'. Some files may have 'score' too; we ignore it.
    """
    extract_dir = _extract_zip_once(lang)
    path = _find_split_file(extract_dir, split)
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            src = ex.get("english word") or ex.get("src") or ""
            tgt = ex.get("native word") or ex.get("tgt") or ""
            if not src or not tgt:
                continue
            rows.append({"src": str(src), "tgt": str(tgt)})
    return rows, path

In [ ]:
def norm_src(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def norm_tgt(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def make_multilingual_rows(rows, lang: str):
    tag = LANG_TOKEN[lang]
    out = []
    for r in rows:
        out.append({
            "lang": lang,
            "src": f"{tag} {norm_src(r['src'])}",
            "tgt": norm_tgt(r["tgt"]),
        })
    return out

def split_train_val(rows, seed=42, val_ratio=0.05, min_val=1000):
    rnd = random.Random(seed)
    rnd.shuffle(rows)
    n = len(rows)
    n_val = max(min_val, int(n * val_ratio))
    val = rows[:n_val]
    train = rows[n_val:]
    return train, val

train_all, val_all, test_all = [], [], []
source_files = {}

for lang in LANGS:
    tr_rows, tr_path = load_aksharantar_jsonl(lang, "train")
    te_rows, te_path = load_aksharantar_jsonl(lang, "test")

    source_files[lang] = {"train": tr_path, "test": te_path}

    tr_multi = make_multilingual_rows(tr_rows, lang)
    te_multi = make_multilingual_rows(te_rows, lang)

    tr_split, va_split = split_train_val(tr_multi, seed=SEED, val_ratio=0.05)
    train_all += tr_split
    val_all += va_split
    test_all += te_multi

random.shuffle(train_all)
random.shuffle(val_all)
random.shuffle(test_all)

print("Counts:", {"train": len(train_all), "val": len(val_all), "test": len(test_all)})
print("Example:", train_all[0])
print("Files:", source_files)

asm.zip:   0%|          | 0.00/4.96M [00:00<?, ?B/s]

hin.zip:   0%|          | 0.00/33.1M [00:00<?, ?B/s]

mar.zip:   0%|          | 0.00/41.8M [00:00<?, ?B/s]

Counts: {'train': 2784008, 'val': 146525, 'test': 27808}
Example: {'lang': 'mar', 'src': '<2mar> anarogyakarak', 'tgt': 'अनारोग्यकारक'}
Files: {'asm': {'train': '/content/aksharantar_zips/asm/asm_train.json', 'test': '/content/aksharantar_zips/asm/asm_test.json'}, 'hin': {'train': '/content/aksharantar_zips/hin/hin_train.json', 'test': '/content/aksharantar_zips/hin/hin_test.json'}, 'mar': {'train': '/content/aksharantar_zips/mar/mar_train.json', 'test': '/content/aksharantar_zips/mar/mar_test.json'}}


In [ ]:
ds = DatasetDict({
    "train": Dataset.from_list(train_all),
    "validation": Dataset.from_list(val_all),
    "test": Dataset.from_list(test_all),
})
ds

DatasetDict({
    train: Dataset({
        features: ['lang', 'src', 'tgt'],
        num_rows: 2784008
    })
    validation: Dataset({
        features: ['lang', 'src', 'tgt'],
        num_rows: 146525
    })
    test: Dataset({
        features: ['lang', 'src', 'tgt'],
        num_rows: 27808
    })
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = tokenizer(
        batch["src"],
        max_length=MAX_SRC,
        truncation=True,
        padding=False,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["tgt"],
            max_length=MAX_TGT,
            truncation=True,
            padding=False,
        )
    inputs["labels"] = labels["input_ids"]
    inputs["lang"] = batch["lang"]  # keep for eval grouping if needed
    return inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

print("Tokenized columns:", tokenized["train"].column_names)

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

Map:   0%|          | 0/2784008 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/146525 [00:00<?, ? examples/s]

Map:   0%|          | 0/27808 [00:00<?, ? examples/s]

Tokenized columns: ['lang', 'input_ids', 'attention_mask', 'labels']


In [ ]:
import torch

# T4 does NOT support bf16 in most cases → fp16 is the right choice
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# Memory/speed helpers
model.gradient_checkpointing_enable()
model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/mt5_xlit_asm_hin_mar",

    # BIG SPEEDUP: no beam-search during training
    evaluation_strategy="no",
    predict_with_generate=False,

    save_strategy="steps",
    save_steps=1000,
    logging_steps=50,

    # faster throughput
    per_device_train_batch_size=32,  # if OOM, set 16
    gradient_accumulation_steps=1,

    learning_rate=8e-4,
    max_steps=4500,        # controls runtime (~60–90 mins depending on batch/seq)
    warmup_steps=100,
    weight_decay=0.01,

    fp16=torch.cuda.is_available() and not use_bf16,
    bf16=use_bf16,

    report_to="none",
    save_total_limit=2,
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

final_dir = "/content/mt5_xlit_asm_hin_mar/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved final model to:", final_dir)

/tmp/ipykernel_843/1041149214.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
50,24.564700
100,9.046500
150,6.219500
200,5.668800
250,5.457900
300,5.359500
350,5.207900
400,5.176100
450,5.088600
500,5.065700


Saved final model to: /content/mt5_xlit_asm_hin_mar/final


In [ ]:
def translit(lang: str, text: str):
    inp = f"{LANG_TOKEN[lang]} {text.strip().lower()}"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    enc = tokenizer([inp], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_length=MAX_TGT, num_beams=5)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

print("hin:", translit("hin", "Saurav"))
print("mar:", translit("mar", "Saurav"))
print("asm:", translit("asm", "Saurav"))

In [ ]:
!ls

aksharantar_zips  mt5_xlit_asm_hin_mar	sample_data


In [ ]:
!zip -r output.zip /content/mt5_xlit_asm_hin_mar/

  adding: content/mt5_xlit_asm_hin_mar/ (stored 0%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/ (stored 0%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/tokenizer.json (deflated 76%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/training_args.bin (deflated 53%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/optimizer.pt (deflated 43%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/model.safetensors (deflated 23%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/tokenizer_config.json (deflated 95%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/spiece.model (deflated 46%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/rng_state.pth (deflated 26%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/config.json (deflated 47%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/trainer_state.json (deflated 79%)
  adding: content/mt5_xlit_asm_hin_mar/checkpoint-4500/scheduler.pt (deflated 62%)
  adding: content/mt5_x

In [ ]:
!pip -q uninstall -y ctranslate2
!pip -q install "ctranslate2==4.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 32.7 MB/s eta 0:00:00


In [ ]:
!ct2-transformers-converter \
  --model /content/mt5_xlit_asm_hin_mar/final \
  --output_dir /content/ct2_mt5_xlit \
  --quantization int8_float16

2026-03-04 17:09:42.326336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772644182.405052   26384 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772644182.431020   26384 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772644182.495840   26384 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772644182.495889   26384 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772644182.495899   26384 computation_placer.cc:177] computation placer alr

In [ ]:
!zip -r output.zip /content/ct2_mt5_xlit/

  adding: content/ct2_mt5_xlit/ (stored 0%)
  adding: content/ct2_mt5_xlit/model.bin (deflated 9%)
  adding: content/ct2_mt5_xlit/config.json (deflated 44%)
  adding: content/ct2_mt5_xlit/shared_vocabulary.json (deflated 70%)


In [ ]:
import os

def folder_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            total += os.path.getsize(os.path.join(root, fn))
    return total / (1024 * 1024)

hf_dir  = "/content/mt5_xlit_asm_hin_mar/final"
ct2_dir = "/content/ct2_mt5_xlit"

print("HF model size (MB):", round(folder_size_mb(hf_dir), 2))
print("CT2 model size (MB):", round(folder_size_mb(ct2_dir), 2))

HF model size (MB): 1164.83
CT2 model size (MB): 293.19


In [ ]:
import time
import torch
import ctranslate2
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

hf_dir  = "/content/mt5_xlit_asm_hin_mar/final"
ct2_dir = "/content/ct2_mt5_xlit"

tokenizer = AutoTokenizer.from_pretrained(hf_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(hf_dir)

MAX_SRC = 24
MAX_TGT = 24

def bench_torch(texts, n_warmup=10, n_runs=100, beams=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()

    # warmup
    for _ in range(n_warmup):
        inp = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC).to(device)
        _ = model.generate(**inp, max_length=MAX_TGT, num_beams=beams)
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    total = 0
    for _ in range(n_runs):
        inp = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC).to(device)
        _ = model.generate(**inp, max_length=MAX_TGT, num_beams=beams)
        total += len(texts)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0
    return {"req": total, "sec": dt, "req_per_s": total/dt}

def bench_ct2(texts, n_warmup=10, n_runs=300, beams=3):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    translator = ctranslate2.Translator(ct2_dir, device=device, compute_type=compute_type)

    enc = tokenizer(texts, return_tensors=None, padding=True, truncation=True, max_length=MAX_SRC)
    input_tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in enc["input_ids"]]

    for _ in range(n_warmup):
        _ = translator.translate_batch(input_tokens, beam_size=beams, max_decoding_length=MAX_TGT)

    t0 = time.time()
    total = 0
    for _ in range(n_runs):
        _ = translator.translate_batch(input_tokens, beam_size=beams, max_decoding_length=MAX_TGT)
        total += len(texts)
    dt = time.time() - t0
    return {"req": total, "sec": dt, "req_per_s": total/dt}

# representative batch
samples = [
    "<2hin> rajesh", "<2mar> priyanka", "<2asm> ananya",
    "<2hin> sachin", "<2mar> rohit", "<2asm> deepak"
] * 4  # batch=24

torch_stats = bench_torch(samples, beams=3)
ct2_stats   = bench_ct2(samples, beams=3)

speed_gain = (ct2_stats["req_per_s"] - torch_stats["req_per_s"]) / torch_stats["req_per_s"] * 100

print("Torch:", torch_stats)
print("CT2  :", ct2_stats)
print(f"Speed gain (%): {speed_gain:.2f}")

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Torch: {'req': 2400, 'sec': 31.509212732315063, 'req_per_s': 76.16819945293713}
CT2  : {'req': 7200, 'sec': 9.491090774536133, 'req_per_s': 758.6061677248991}
Speed gain (%): 895.96


In [ ]:
import numpy as np

def cer(ref: str, hyp: str) -> float:
    r, h = ref, hyp
    n, m = len(r), len(h)
    if n == 0:
        return 0.0 if m == 0 else 1.0
    dp = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): dp[i,0] = i
    for j in range(m+1): dp[0,j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+cost)
    return dp[n,m] / n

def eval_ct2(raw_ds, max_samples=3000, beams=3):
    if max_samples and len(raw_ds) > max_samples:
        raw_ds = raw_ds.select(range(max_samples))

    device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    translator = ctranslate2.Translator(ct2_dir, device=device, compute_type=compute_type)

    preds, refs, langs = [], [], []
    bs = 64
    for i in range(0, len(raw_ds), bs):
        batch = raw_ds[i:i+bs]
        enc = tokenizer(batch["src"], return_tensors=None, padding=True, truncation=True, max_length=MAX_SRC)
        input_tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in enc["input_ids"]]
        outs = translator.translate_batch(input_tokens, beam_size=beams, max_decoding_length=MAX_TGT)
        hyp = [tokenizer.convert_tokens_to_string(o.hypotheses[0]).strip() for o in outs]
        preds.extend(hyp)
        refs.extend([t.strip() for t in batch["tgt"]])
        langs.extend(batch["lang"])

    # metrics
    acc = float(np.mean([p == r for p, r in zip(preds, refs)]))
    cer_mean = float(np.mean([cer(r, p) for p, r in zip(preds, refs)]))
    return {"accuracy": acc, "cer": cer_mean}

print(eval_ct2(ds["test"], max_samples=3000, beams=3))

{'accuracy': 0.18133333333333335, 'cer': 0.3661872554403514}
